# Phase 0 + Phase 1 on Kaggle T4 x2 (or Colab T4)

**Kaggle:** Settings → Accelerator **GPU T4 x2**, Internet **On**. Work is split across both cards: one runs
the engine-level A/Bs, the other runs kernel microbenchmarks and the profile. On a single-GPU Colab the same
cells run everything sequentially on GPU 0.

Every benchmark is its own process pinned to one GPU with `CUDA_VISIBLE_DEVICES`; this kernel never loads a model
except in the short hook check, which releases it. Results land in `results/t4/<date>_<commit>/`. Commit or download
that folder before the session ends.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.used,clocks.max.sm,power.limit --format=csv
import os, subprocess
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
GPUS = list(range(int(subprocess.check_output(["nvidia-smi","-L"], text=True).count("GPU ")) or 1))
print("kaggle" if ON_KAGGLE else "colab", "| gpus:", GPUS)

In [ ]:
os.chdir("/kaggle/working" if ON_KAGGLE else "/content")
if not os.path.isdir("fol"):
    !git clone -b t4-phase0 https://github.com/Vaibhav7711/full-inference-engine.git fol
%cd fol
!git pull -q && git log -1 --oneline
%env TOKENIZERS_PARALLELISM=false
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

In [ ]:
!bash scripts/setup_kaggle.sh
!python scripts/colab_preflight.py

In [ ]:
import datetime, pathlib
SHA = subprocess.check_output(["git","rev-parse","--short","HEAD"], text=True).strip()
RUN = f"results/t4/{datetime.date.today():%Y%m%d}_{SHA}"
pathlib.Path(RUN).mkdir(parents=True, exist_ok=True)
%env RUN=$RUN
print(RUN)

## Runner

`run_queues` takes one job list per GPU and runs the lists concurrently, each list sequentially, each job pinned to
its GPU with `CUDA_VISIBLE_DEVICES`. Output goes to `$RUN/logs/<name>.log`; the tail of each log is printed when
everything finishes. On one GPU all lists are concatenated onto GPU 0.

In [ ]:
import subprocess, threading, time, os, pathlib

def gpu_used_mb(index=0):
    return int(subprocess.check_output(["nvidia-smi", f"--id={index}", "--query-gpu=memory.used",
                                        "--format=csv,noheader,nounits"], text=True))

def assert_gpus_free(limit_mb=600):
    for g in GPUS:
        used = gpu_used_mb(g)
        assert used < limit_mb, f"GPU {g}: {used} MiB in use - restart the kernel (Run -> Restart) before benchmarking"
    print("GPUs free:", {g: f"{gpu_used_mb(g)} MiB" for g in GPUS})

def run_queues(queues, tail=8):
    """queues: list of [(name, command), ...]; queue i runs on GPU i (all on GPU 0 if only one)."""
    logs = pathlib.Path(RUN) / "logs"; logs.mkdir(exist_ok=True)
    if len(GPUS) == 1:
        queues = [[job for queue in queues for job in queue]]
    results = {}
    def worker(gpu, queue):
        for name, command in queue:
            env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
            started = time.perf_counter()
            with open(logs / f"{name}.log", "w") as log:
                code = subprocess.call(command, shell=True, stdout=log, stderr=subprocess.STDOUT, env=env, cwd=os.getcwd())
            results[name] = (gpu, code, time.perf_counter() - started)
            print(f"[gpu{gpu}] {name}: exit {code} in {results[name][2]/60:.1f} min", flush=True)
    threads = [threading.Thread(target=worker, args=(GPUS[i], queue)) for i, queue in enumerate(queues)]
    for t in threads: t.start()
    for t in threads: t.join()
    for name, (gpu, code, seconds) in results.items():
        print(f"\n===== {name} (gpu{gpu}, exit {code}) =====")
        print("".join(open(logs / f"{name}.log").readlines()[-tail:]))
    failed = [name for name, (_, code, _) in results.items() if code]
    assert not failed, f"failed: {failed} - see {logs}"

## Phase 0 — gates

**0.1 GPU correctness**, split by subsystem across the two cards. The parked speculative suite is excluded: its FP32
parity test loads Qwen3-1.7B in fp32 and needs ~10 GB by itself.

In [ ]:
assert_gpus_free()
PYTEST = "python -m pytest -q -m cuda -p no:cacheprovider"
run_queues([
    [("gate_kernels_cache_server",
      f"{PYTEST} tests/kernels tests/cache tests/correctness tests/server")],
    [("gate_batching_reliability",
      f"{PYTEST} tests/batching tests/reliability --ignore=tests/batching/test_batched_speculative.py")],
], tail=4)

**0.2 Hooks work.** Warm-up must capture every bucket in both decode regimes (`graphs == 2 × buckets`) and compile both
prefill paths; one instrumented decode step must report all phases; `host_stage_ms` should be well under 1 ms at batch 8.
This is the only cell that loads a model in this kernel, and it releases it.

In [ ]:
import time, gc, torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from engine.model import load_model
from engine.batching.continuous_batching import ContinuousBatchingEngine
from benchmarks.common import device_clock_record, git_record

loaded = load_model("Qwen/Qwen3-0.6B")
engine = ContinuousBatchingEngine(loaded.model, loaded.tokenizer, loaded.device,
                                  num_blocks=1024, max_active=8, cuda_graph_batch_sizes=(2, 4, 8))
t0 = time.perf_counter(); summary = engine.warmup(); print(f"warmup {time.perf_counter()-t0:.1f}s ->", summary)
print("captured:", sorted(engine._decode_graphs))
assert summary["graphs"] == 2 * 3, f"expected every (bucket, block_n) pair: {summary}"
assert summary["prefill_sdpa_calls"] and summary["prefill_chunked_calls"], summary
engine.instrument = True
engine.generate(["Explain KV caching in one sentence."] * 8, max_new_tokens=8)
print("last step phases (ms):", {k: round(v, 3) for k, v in engine.last_step_timing.items()})
assert {"host_stage_ms", "decode_gpu_ms", "sync_ms"} <= set(engine.last_step_timing)
print("git:", git_record()); print("clocks:", device_clock_record())
del engine, loaded; gc.collect(); torch.cuda.empty_cache()
print(f"allocated after release: {torch.cuda.memory_allocated()/1e6:.0f} MB")

**0.3 Token gate + interleaving plumbing check** (two short runs; not a result).

In [ ]:
run_queues([[("phase0_smoke",
  "python -m benchmarks.reliability.ab --setting cuda_graphs --repeats 2 --duration 4 --out $RUN/phase0_smoke_cuda_graphs.json")]], tail=12)

## Phase 1 — re-baseline, both GPUs

**GPU 0** runs the engine A/Bs one after another: they are host-sensitive (the step split includes CPU staging), so they
never share a card and the interleaved order handles thermal drift. **GPU 1** runs the kernel microbenchmarks and the
profile in parallel: those are timed with CUDA events on their own card and are insensitive to the other process.
Kaggle gives four host cores; two Python processes fit. Expect ~25 minutes wall-clock.

Each A/B: 5 interleaved runs per arm, `chat` profile unless the setting only binds on longer prompts;
`--cuda-graphs` puts both arms on the graphed decode path so the prefill setting is the only difference.

In [ ]:
assert_gpus_free()
AB = "python -m benchmarks.reliability.ab --repeats 5"
engine_abs = [
    ("ab_cuda_graphs_chat",   f"{AB} --setting cuda_graphs   --prompt-profile chat               --out $RUN/ab_cuda_graphs_chat.json"),
    ("ab_prefill_kernel_chat",f"{AB} --setting prefill_kernel --prompt-profile chat --cuda-graphs --out $RUN/ab_prefill_kernel_chat.json"),
    ("ab_prefill_chunk_chat", f"{AB} --setting prefill_chunk  --prompt-profile chat --cuda-graphs --out $RUN/ab_prefill_chunk_chat.json"),
    ("ab_prefix_cache_chat",  f"{AB} --setting prefix_cache   --prompt-profile chat --cuda-graphs --out $RUN/ab_prefix_cache_chat.json"),
    ("ab_kv_dtype_long",      f"{AB} --setting kv_dtype --prompt-profile long --cuda-graphs --num-blocks 512 --out $RUN/ab_kv_dtype_long.json"),
]
kernel_jobs = [
    ("roofline",       "python -m benchmarks.kernels.roofline --out $RUN/roofline.json"),
    ("profile_decode", "python -m benchmarks.batching.profile_continuous_decode --concurrency 8 --output $RUN/profile_continuous_decode.json"),
    ("mlp_fusion_ab",  "python -m benchmarks.batching.mlp_gate_up_fusion_ab --rounds 7 --output $RUN/mlp_gate_up_fusion_ab.json"),
    ("prefill_tiles",  "python -m benchmarks.kernels.prefill_attention_ab --sweep-tiles --out $RUN/prefill_attention_sweep.json"),
]
run_queues([engine_abs, kernel_jobs], tail=6)

## Summary

One block per A/B: the verdicts on the metrics that decide things. `unresolved` means inside run-to-run spread — record it
as unresolved, not as a small win. `prefill_kernel` decides the `tiled_prefill` default; any non-`kv_dtype` A/B with
`tokens identical=False` is a kernel bug, not a result.

In [ ]:
import json, glob
for path in sorted(glob.glob(f"{RUN}/ab_*.json")):
    d = json.load(open(path)); a, b = list(d["arms"])
    print(f"\n{os.path.basename(path)} [{b} vs {a}] tokens identical={d['token_identity']['identical']} "
          f"SM MHz before/after: {d['clocks_before'].get('clocks.sm')}/{d['clocks_after'].get('clocks.sm')}")
    for m in ("step_timing.expected_gap_ms", "step_timing.decode_step_p50_ms", "step_timing.prefill_step_p50_ms",
              "latency.ttft_p50", "latency.itl_p99", "step_timing.host_stage_ms_p50", "step_timing.decode_gpu_ms_p50"):
        base = d["arms"][a]["summary"].get(m, {})
        print(f"  {m:36s} base={base.get('median', float('nan')):8.3f}  {d['comparison'].get(m, '-')}")

## Save before the session dies

In [ ]:
!git add $RUN && git -c user.name=t4-runner -c user.email=t4@local commit -q -m "T4 phase 0/1 results $RUN" && git log -1 --oneline
# !git push https://<TOKEN>@github.com/Vaibhav7711/full-inference-engine.git HEAD:t4-phase0
!zip -qr ../t4_results.zip $RUN && ls -la ../t4_results.zip $RUN